# Pipeline de filtrado — PRs CI/CD rechazados (AIDev)

F0 universo → F1 al menos un archivo CI/CD → F2 rechazado → export CSV.


## 1. Cargar datasets

In [1]:
import pandas as pd

BASE = "hf://datasets/hao-li/AIDev/"

pull_request      = pd.read_parquet(BASE + "pull_request.parquet")
pr_commit_details = pd.read_parquet(BASE + "pr_commit_details.parquet")

print(f"pull_request:      {pull_request.shape}")
print(f"pr_commit_details: {pr_commit_details.shape}")


pull_request:      (33596, 14)
pr_commit_details: (711923, 14)


## 2. Filtrado 1 — el PR modifica al menos un archivo CI/CD

In [2]:
CI_PATTERNS = [
    r"\.github/workflows/",
    r"\.gitlab-ci\.ya?ml$",
    r"azure-pipelines\.ya?ml$",
    r"\.circleci/config\.ya?ml$",
    r"Jenkinsfile(\.[^/]+)?$",
    r"\.travis\.ya?ml$",
    r"bitbucket-pipelines\.ya?ml$",
    r"\.drone\.ya?ml$",
    r"buildkite\.ya?ml$",
    r"appveyor\.ya?ml$",
    r"semaphore\.ya?ml$",
    r"codefresh\.ya?ml$",
]
CI_REGEX = "|".join(CI_PATTERNS)

ci_files = pr_commit_details[
    pr_commit_details["filename"].str.contains(CI_REGEX, case=False, na=False, regex=True)
].copy()

f1_pr_ids = set(ci_files["pr_id"].unique())
f1_prs = pull_request[pull_request["id"].isin(f1_pr_ids)].copy()

print(f"F1 - filas (PR, archivo CI/CD): {len(ci_files):,}")
print(f"F1 - PRs unicos con >=1 archivo CI/CD: {len(f1_prs):,}")


/tmp/ipykernel_26718/2960148305.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  pr_commit_details["filename"].str.contains(CI_REGEX, case=False, na=False, regex=True)


F1 - filas (PR, archivo CI/CD): 7,866
F1 - PRs unicos con >=1 archivo CI/CD: 2,376


## 3. Filtrado 2 — PR rechazado (`state == 'closed' AND merged_at IS NULL`)

In [3]:
rejected_prs = f1_prs[(f1_prs["state"] == "closed") & (f1_prs["merged_at"].isna())].copy()
print(f"F2 - PRs rechazados: {len(rejected_prs):,}")


F2 - PRs rechazados: 454


## 4. Filtrado 3 — una fila por PR (eliminar PRs repetidos)

In [4]:
# Agregar los archivos CI/CD por PR para tener una unica fila por pull request.
# Las columnas numericas se suman, las textuales se concatenan con separador.
rejected_ci_files = ci_files[ci_files["pr_id"].isin(rejected_prs["id"])].copy()

def join_unique(values, sep=" || ", limit=None):
    items = [str(v) for v in values if v is not None and str(v) != "nan"]
    seen = []
    for it in items:
        if it not in seen:
            seen.append(it)
        if limit and len(seen) >= limit:
            break
    return sep.join(seen)

f3_files_per_pr = (rejected_ci_files.groupby("pr_id")
                   .agg(ci_filenames=("filename", lambda x: join_unique(x)),
                        ci_n_files=("filename", "nunique"),
                        ci_statuses=("status", lambda x: join_unique(x)),
                        ci_additions=("additions", "sum"),
                        ci_deletions=("deletions", "sum"),
                        ci_changes=("changes", "sum"),
                        ci_commit_messages=("message", lambda x: join_unique(x, limit=5)),
                        ci_patches=("patch", lambda x: join_unique(x, limit=10)))
                   .reset_index())

print(f"F3 - filas (1 por PR): {len(f3_files_per_pr):,}")
print(f"F3 - PRs unicos: {f3_files_per_pr['pr_id'].nunique():,}")


F3 - filas (1 por PR): 454
F3 - PRs unicos: 454


## 5. Filtrado 4 — PRs con exactamente un archivo CI/CD

In [5]:
f4_files_per_pr = f3_files_per_pr[f3_files_per_pr["ci_n_files"] == 1].copy()
rejected_prs_f4 = rejected_prs[rejected_prs["id"].isin(f4_files_per_pr["pr_id"])].copy()

print(f"F4 - PRs con exactamente 1 archivo CI/CD: {len(f4_files_per_pr):,}")
print(f"     (descartados por tener >1 archivo CI/CD: {len(f3_files_per_pr) - len(f4_files_per_pr):,})")

F4 - PRs con exactamente 1 archivo CI/CD: 269
     (descartados por tener >1 archivo CI/CD: 185)


## 6. Exportar CSV

In [6]:
from pathlib import Path
OUT = Path("data"); OUT.mkdir(exist_ok=True)

# Una fila por PR rechazado con exactamente 1 archivo CI/CD
pr_cols = rejected_prs_f4.rename(columns={"id": "pr_id"})[
    ["pr_id", "number", "title", "state", "created_at", "closed_at",
     "merged_at", "repo_url", "html_url", "agent", "body"]
]
export = pr_cols.merge(f4_files_per_pr, on="pr_id", how="left")

export.to_csv(OUT / "ci_cd_pr_filtered.csv", index=False, encoding="utf-8-sig")
print(f"data/ci_cd_pr_filtered.csv: {len(export):,} filas, {export['pr_id'].nunique():,} PRs unicos")

data/ci_cd_pr_filtered.csv: 269 filas, 269 PRs unicos
